# Crop Recommendation System — Cell-wise Model Training
**Algorithm:** Decision Tree Classifier  
**Dataset:** `dataset/Crop_recommendation_cleaned.csv`

Run the cells from top to bottom. The notebook covers loading, validation, cleaning, exploratory checks, splitting, preprocessing, training, evaluation, and saving the model.

## Cell 1 — Import libraries

In [ ]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Cell 2 — Load the dataset

In [ ]:
DATA_PATH = "dataset/Crop_recommendation_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## Cell 3 — Validate columns and data types

In [ ]:
df.columns = df.columns.str.strip()

required_columns = [
    "N", "P", "K", "temperature", "humidity", "ph", "rainfall", "label"
]

missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("All required columns are present.")
print("\nData types:")
display(df.dtypes)

## Cell 4 — Check missing values and duplicates

In [ ]:
print("Missing values per column:")
display(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())
print("Blank crop labels:", df["label"].astype(str).str.strip().eq("").sum())

## Cell 5 — Clean the dataset

In [ ]:
# Clean whitespace in column names and text values
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].str.strip()

feature_columns = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]

# Convert feature columns to numeric; invalid values become NaN
for col in feature_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before_rows = len(df)

# Remove rows missing any required feature or target
df = df.dropna(subset=feature_columns + ["label"]).copy()

# Remove blank labels
df = df[df["label"].astype(str).str.strip().ne("")].copy()

# Remove exact duplicate records
df = df.drop_duplicates().copy()

# Reject negative N, P, K, and rainfall values
nonnegative_columns = ["N", "P", "K", "rainfall"]
invalid_mask = (df[nonnegative_columns] < 0).any(axis=1)
df = df.loc[~invalid_mask].copy()

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", len(df))
print("Rows removed:", before_rows - len(df))
print("Missing values remaining:", int(df.isna().sum().sum()))
print("Duplicate rows remaining:", int(df.duplicated().sum()))

## Cell 6 — Inspect class distribution and summary statistics

In [ ]:
print("Number of crop classes:", df["label"].nunique())
display(df["label"].value_counts().sort_index())
display(df[feature_columns].describe().T)

## Cell 7 — Separate features (X) and target (y)

In [ ]:
X = df[feature_columns]
y = df["label"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

## Cell 8 — Split into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## Cell 9 — Preprocess using StandardScaler

In [ ]:
# Fit the scaler only on training data to avoid test-data leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

## Cell 10 — Train the Decision Tree model

In [ ]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train_scaled, y_train)

print("Model training completed.")

## Cell 11 — Evaluate the model

In [ ]:
pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(y_test, pred, zero_division=0))

## Cell 12 — Save model, scaler, and cleaned dataset

In [ ]:
os.makedirs("model", exist_ok=True)

joblib.dump(model, "model/crop_model.pkl")
joblib.dump(scaler, "model/scaler.pkl")
df.to_csv("dataset/Crop_recommendation_cleaned.csv", index=False)

print("Saved model/crop_model.pkl")
print("Saved model/scaler.pkl")
print("Saved dataset/Crop_recommendation_cleaned.csv")

## Notes
- The model expects features in this exact order: `N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`.
- The Flask app must apply the saved scaler before calling `model.predict()`.
- Decision Trees generally do not require scaling, but this project keeps `StandardScaler` to match the requested workflow and existing app setup.